In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'tensorflow', 'keras', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'keras': 'keras==3.14.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "061a82116be00c585cab41d5b2a790e2a4f96d44"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/061a82116be00c585cab41d5b2a790e2a4f96d44/d2l"
for name in ('__init__.py', 'tensorflow.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Reproducibility and Inspection

Suppose we change one line of a training script and the validation accuracy
of a Fashion-MNIST classifier rises by 0.3 points. Before we credit the
change, we need to know how much two runs of the unchanged script differ.
For softmax regression we measure the answer in
that section: five runs that differ only in their seed end
one to four points apart, because each run draws different initial weights
and visits the examples in a different order. A gain of 0.3 points is well
inside that range.

A run that fails poses a second problem of the same kind. If the loss turns
into NaN after hours of training, we want to repeat the run exactly up to
that point and then look inside the model to see which layer produced the
first non-finite number.

This section first makes runs repeatable. Every random draw comes from a
generator whose sequence of numbers is fixed by a seed, so seeding the
generators fixes the draws. That is the easy part. Repeating a run bit for
bit also requires that the arithmetic adds numbers in a fixed order, which
a GPU does not promise. The section then looks inside running models with
*hooks*, functions that observe each layer of an unmodified model as it
computes. The two halves serve one purpose: a measurement taken inside a
model can be compared across two versions of the code only if the runs
differ in nothing but that code. On a first reading, the subsections on
generator objects and on loader workers can be skimmed; they matter once a
data pipeline draws random numbers of its own.

In [ ]:
import math
import numpy as np
import tensorflow as tf
from d2l import tensorflow as d2l

In [ ]:
import random

## Seeds and Random Number Generators

Randomness enters a training run in more places than one might list at
first: initialization draws every weight (that section), dropout
draws a new mask at every step, the data loader shuffles the examples
differently in every epoch, data augmentation draws crops and flips, and
parallel loader workers do some of this work in separate processes. Each
draw comes from a *pseudorandom number generator*, a deterministic algorithm
whose sequence of outputs is fixed by its starting state, the *seed*.
Seeding every generator that a run consults therefore fixes every draw.

### Global Seeds

A framework's random functions either consult a global generator or take
their state as an argument. Which of the two a framework does decides how a
run is seeded. To see a seed at work, the function below trains a small
network for five steps on random data and draws everything, weights and
data, from the one source that its seed fixes.

`tf.keras.utils.set_random_seed(k)` seeds Python's `random` module, NumPy's
global generator, and TensorFlow's global generator in one call, the three
that a typical training script consults. It does not reach generator objects
that carry their own state (`tf.random.Generator`, `np.random.default_rng`).

In [ ]:
def train_once(seed):
    tf.keras.utils.set_random_seed(seed)  # seeds Python, NumPy, and TF
    with tf.device('/CPU:0'):  # see Deterministic Arithmetic below
        net = tf.keras.Sequential([
            tf.keras.layers.Dense(32, activation='relu'),
            tf.keras.layers.Dense(1)])
        opt = tf.keras.optimizers.SGD(learning_rate=0.1)
        X, y = tf.random.normal((128, 20)), tf.random.normal((128, 1))
        for _ in range(5):
            with tf.GradientTape() as tape:
                loss = tf.reduce_mean((net(X) - y) ** 2)
            grads = tape.gradient(loss, net.trainable_variables)
            opt.apply_gradients(zip(grads, net.trainable_variables))
        return float(loss), tf.identity(net.layers[0].kernel)

loss_a, w_a = train_once(seed=0)
loss_b, w_b = train_once(seed=0)
loss_c, _ = train_once(seed=1)
print('same seed, identical loss and weights:',
      loss_a == loss_b, bool(tf.reduce_all(w_a == w_b)))
print('different seed:', loss_a, 'vs', loss_c)

The two runs with seed 0 agree *bitwise*, down to the last bit of the loss
and of every weight: same initial weights, same data, same gradients, same
operations in the same order. (The order is the one thing a seed does not
control; we return to it in that section.) The
run with seed 1 ends with a different loss.

We placed the TensorFlow run on the CPU. On a GPU, the same seeded code
gives equal losses but, in some runs, weights that differ in their last
bits.

### Generator Objects

A single global generator couples all the code that draws from it. Inserting
one extra random call, a new layer's initialization or a stray draw while
debugging, shifts every number drawn after it, so the data split, the
initialization, and the dropout masks all change at once. A *generator
object* is a private stream with its own seed, which unrelated draws cannot
shift.

A `tf.random.Generator` is such a stream. Here a random permutation, say a
split of a dataset into training and validation examples, stays fixed no
matter what else draws from the global generator in between (the generator
has no permutation method, so we sort uniform random numbers):

In [ ]:
g = tf.random.Generator.from_seed(42)
split = tf.argsort(g.uniform((10,)))  # a random permutation from g's stream
_ = tf.random.normal((1000,))  # unrelated draws from the global generator
split_again = tf.argsort(tf.random.Generator.from_seed(42).uniform((10,)))
print(bool(tf.reduce_all(split == split_again)), split.numpy())

Sampling methods live on the generator itself (`g.normal`, `g.uniform`), so
code that must own its randomness takes a `Generator` argument. The input
pipeline instead takes a seed directly, which we use next.

### Data Loader Workers

Data loaders often prepare minibatches in parallel *worker* processes. A
worker that draws random numbers, for data augmentation for instance, needs
a stream of its own. Two failures are common, and neither raises an error.
If every worker starts as a copy of the main process, as with the *fork*
start method, the workers inherit the same generator state and apply the
same "random" augmentations. If each worker seeds itself from the operating
system's entropy, no two runs agree.

`tf.data` avoids both failures by design. It runs its parallel work in
threads of one process, so no worker inherits a copied generator, and its
randomness is an explicit argument, `Dataset.shuffle(buffer_size,
seed=...)`. With the default `reshuffle_each_iteration=True`, a seeded
pipeline produces a *different* order in each epoch, but the same sequence
of orders every time the pipeline is built:

In [ ]:
ds = tf.data.Dataset.range(8).shuffle(8, seed=0).batch(4)
print([b.numpy().tolist() for b in ds])  # epoch 1
print([b.numpy().tolist() for b in ds])  # epoch 2: reshuffled, still seeded
ds = tf.data.Dataset.range(8).shuffle(8, seed=0).batch(4)
print([b.numpy().tolist() for b in ds])  # rebuilt pipeline: epoch 1 again

`reshuffle_each_iteration=False` keeps one order for every epoch, and
leaving `seed` unset draws new orders in each run. Parallel `map` and
`interleave` also accept `deterministic=False`, which passes elements on in
the order they finish, for speed; the default, `True`, keeps the order of
the output fixed.

### Randomness as a Value

The worker failures above arise because the state of a generator is copied
or shared without appearing in the program's data flow. JAX makes randomness
a value instead: every random function takes a key, the same key gives the
same numbers, and independent streams come from splitting a key. A key can
still be reused by mistake, but the reuse is visible in the code. The cost
is that every function that draws random numbers must accept a key and pass
split keys on. A global seed is the convenient form of the same contract: one
implicit key that all code shares. Generator objects recover the explicit
form where it matters.

TensorFlow offers both forms: the stateful functions above, and stateless
ones that take the key as an argument, such as
`tf.random.stateless_normal(shape, seed=[k1, k2])`.

### The Spread across Seeds

We can now measure the spread that the opening asked about. The function
below trains the softmax regression of that section on
Fashion-MNIST for three epochs on a GPU, seeds every generator that the run
consults, and returns the final validation accuracy and the trained
weights. We run it with the seeds 0 to 4.

`set_random_seed` fixes the initialization and the shuffle order of the
`tf.data` pipeline.

In [ ]:
def train_classifier(seed, data):
    tf.keras.utils.set_random_seed(seed)  # initialization and shuffle order
    model = d2l.SoftmaxRegression(num_outputs=10, lr=0.1)
    trainer = d2l.Trainer(max_epochs=3, num_gpus=1)
    trainer.board.display = False
    trainer.fit(model, data)
    return (trainer.history['val_acc'][-1],
            tf.identity(model.net.layers[1].kernel))

data = d2l.FashionMNIST(batch_size=256)
runs = [train_classifier(seed, data) for seed in range(5)]
accs = 100 * np.array([acc for acc, _ in runs])
print('validation accuracy (%):', ' '.join(f'{a:.1f}' for a in accs))
print(f'mean {accs.mean():.1f}, max - min {np.ptp(accs):.1f} points')

The five runs differ only in their seeds, yet their final accuracies lie
one to four points apart, around 81%. A gain of 0.3 points, measured on one
run of each variant, is well inside that spread. To credit a change we
compare the mean over several seeds per variant, together with the spread.

The seed is all that separated these runs. To confirm it, we repeat the
run with seed 0 and compare it with the first one, accuracy and every
weight:

In [ ]:
acc, w = train_classifier(0, data)
diff = float(tf.reduce_max(tf.abs(w - runs[0][1])))
print('same accuracy:', acc == runs[0][0],
      '| same weights:', bool(tf.reduce_all(w == runs[0][1])),
      f'| largest difference {diff:.2g}')

The rerun reproduces the accuracy of the first run exactly. Whether every
weight agrees as well depends on the order in which the GPU adds numbers,
which a seed does not fix (that section).

Here the weights agree bitwise in some reruns and differ in their last bits,
by less than $10^{-6}$, in others: on a GPU, TensorFlow adds the terms of
the gradient of a bias in a varying order.

## Deterministic Arithmetic

Seeding fixes which numbers the program draws. It does not fix how the
arithmetic combines them. Floating-point addition is not associative: every
sum is rounded to the nearest representable number, so adding the same
numbers in a different order can give a different result.

Three float32 numbers suffice to show it. Near $10^8$, adjacent float32
numbers are 8 apart (that section), so $-10^8 + 1$ rounds back to
$-10^8$:

In [ ]:
a, b, c = [tf.constant(v, dtype=tf.float32) for v in (1e8, -1e8, 1.0)]
print(float((a + b) + c), float(a + (b + c)))  # same numbers, regrouped

Regrouping changes the result from 1 to 0. In a long sum the effect is
smaller but of the same kind: adding a million random numbers forward and
backward usually gives results that differ in their last bits.

On a CPU, the order in which one operation adds its numbers is usually fixed
for a given number of threads, so a seeded run repeats. On a GPU, many
threads add partial sums at the same time, and some operations combine them
with *atomic additions*, which commit each partial sum in whatever order the
threads happen to finish. Examples are the gradients of an embedding lookup
or of a scatter operation, some convolution gradients, and some reductions.
Two seeded runs on the same machine can then differ in the last bits of
their weights, and over many training steps such differences grow until the
loss curves differ visibly. The remedy is a switch that replaces such
operations with deterministic ones, which add in a fixed order and are often
slower. The cell below shows both. A *scatter-add* sums a million random
numbers into ten bins, each number into the bin its index names; we run it
twice on a GPU and compare the two results bit by bit.

`tf.config.experimental.enable_op_determinism()` is the switch. Operations
that have a deterministic implementation use it, and operations without one
raise `tf.errors.UnimplementedError`. The switch belongs at the start of the
program, and nothing turns it off again short of a new process, so the cell
runs the scatter-add `tf.math.unsorted_segment_sum` first without it, then
with it:

In [ ]:
if tf.config.list_physical_devices('GPU'):
    src = tf.random.normal((2**20,))  # a million numbers
    idx = tf.random.uniform((2**20,), maxval=10, dtype=tf.int32)  # their bins
    for deterministic in (False, True):
        if deterministic:
            tf.config.experimental.enable_op_determinism()
        a, b = [tf.math.unsorted_segment_sum(src, idx, 10) for _ in range(2)]
        print(f'deterministic {deterministic!s:5}: bitwise equal '
              f'{bool(tf.reduce_all(a == b))}, largest difference '
              f'{float(tf.reduce_max(tf.abs(a - b))):.2g}')
else:
    print('no GPU: on a CPU this sum adds in a fixed order')

The two scatter-adds without the switch differ in their last bits, by a
few thousandths on bin sums of a few hundred; the deterministic computation
agrees with itself bitwise.

With the switch on, random operations must have a seed, since an operation
that seeds itself from the operating system is nondeterministic too. We can
demonstrate the rule by clearing the global seed:

In [ ]:
tf.config.experimental.enable_op_determinism()  # no-op if already on
tf.random.set_seed(None)  # forget the global seed for a moment
try:
    tf.random.normal((2,))
except RuntimeError as e:
    print(str(e).split('.')[0])
# determinism stays on and needs a seed, though not a particular one
tf.keras.utils.set_random_seed(random.SystemRandom().getrandbits(31))

Bitwise agreement holds only for one machine, one library version, and one
configuration of these switches; no framework promises it across releases,
across hardware, or between CPU and GPU. Bitwise reproducibility is
therefore a tool for debugging: it lets us find the step at which two runs
first differ. The scientific goal is weaker and more robust. A conclusion,
such as "the change raises accuracy", should hold across seeds, so we report
the mean and the spread over several runs rather than one run, as for the
five seeds of that section (about 81%, with one to four
points between the best and the worst run). As with the
number formats of that section, a conclusion that changes with the
seed or with the last bits of the arithmetic needs more evidence.

## Model Inspection with Hooks

A repeatable run tells us that a problem will recur, not where it arises.
To find out, we observe what each layer computes during the run. Editing the
`forward` method of every layer to print or store its output is tedious, and
impossible for a model that came from a library or a checkpoint whose code
we do not want to change. A *hook* is a function that the framework calls at
a fixed point of each layer's computation, from outside the layer's code.

Keras offers no hooks: nothing can be attached to an unmodified model to
observe it. Two idioms reach the same measurements. In a *functional* model,
built by calling layers on symbolic inputs, every intermediate tensor is an
object we can name, so a second `tf.keras.Model` over the same graph can
return any of them as an output; it shares all weights with the first. When
we own the model's code, an overridden `call` can record the values we want.
the figure shows where a hook would run; in TensorFlow the
observation points must be built into the model.

![The `__call__` wrapper as a pipeline: input flows through pre-hooks, then forward, then hooks, to the output, with the two hook stages dashed and orange against forward's solid blue, and a side arrow from the hooks stage to an observer that can capture, check, or modify.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-hooks.svg)

As a test model we use a stack of eight residual blocks of width 64 between
an input layer with 20 inputs and an output layer with 10 outputs. The block
is the one of that section: it adds the output of its
*body*, a linear layer, a ReLU, and a second linear layer, to its input,
$\mathbf{h} = \mathbf{x} + \textrm{body}(\mathbf{x})$. The weights start
from Xavier initialization (that section) and the biases at zero.
We do not seed this model. Its weights are drawn afresh in every run, which
is why the numbers below are quoted as ranges.

In [ ]:
def dense(num_outputs, **kwargs):  # Xavier weights, zero biases
    return tf.keras.layers.Dense(
        num_outputs, kernel_initializer='glorot_uniform', **kwargs)

class ResidualBlock(tf.keras.layers.Layer):
    def __init__(self, num_hiddens, **kwargs):
        super().__init__(**kwargs)
        self.body = tf.keras.Sequential([
            dense(num_hiddens, activation='relu'), dense(num_hiddens)])

    def call(self, X):
        return X + self.body(X)

inputs = tf.keras.Input(shape=(20,))
taps = [dense(64)(inputs)]  # keep every intermediate tensor
for _ in range(8):
    taps.append(ResidualBlock(64)(taps[-1]))
outputs = dense(10)(taps[-1])
net = tf.keras.Model(inputs, outputs)

### Activation Statistics

The initialization experiments of that section measured the
standard deviation of the activations at every depth. With hooks, the same
measurement takes a few lines on an unmodified model. We record the standard
deviation of each layer's output on a minibatch of 256 standard normal
inputs.

In [ ]:
probe = tf.keras.Model(inputs, taps + [outputs])  # same layers, same weights
X = tf.random.normal((256, 20))
names = ['Dense'] + ['ResidualBlock'] * 8 + ['Dense']
for name, out in zip(names, probe(X)):
    print(f'{name:15s} std {float(tf.math.reduce_std(out)):.2f}')

The standard deviation grows from block to block, from about 0.7 after the
input layer to roughly 3 to 4 after the eighth block. Both numbers can be
predicted. Xavier initialization gives the input layer, with 20 inputs and
64 outputs, weights of variance $2/84$, so its output has variance
$20 \cdot 2/84 \approx 0.48$ and standard deviation 0.69. Each block then
adds its body's output to its input, and the two are nearly uncorrelated, so
their variances add. The body's output has about half the variance of its
input: the first linear layer keeps the variance, the ReLU halves the mean
square, and the second linear layer keeps it again. The variance therefore
grows by a factor of about 1.5 per block, and the standard deviation by
about $\sqrt{1.5} \approx 1.22$. Eight blocks should multiply 0.69 by
$1.5^4 \approx 5$, which gives 3.5, and the table agrees to within the
scatter of one random draw. This is the calculation of
that section with a factor of 1.5 where He initialization gave
2, and it is the reason why the bodies of deep residual networks are often
initialized with smaller weights.

Nothing needs to be detached or removed: `probe` shares the original model's
layers and weights, its outputs are ordinary tensors outside any gradient
tape, and no observer stays attached to the model. The method requires a
functional model; a subclassed model has no symbolic intermediate tensors
to return, so its `call` must record the values itself.

### A NaN Finder

When a loss becomes NaN at step 40,000, the useful question is which layer
produced the first non-finite value. A hook on every layer that checks its
output for infinities and NaNs answers it in one forward pass: the first
hook to fire names the layer, because hooks fire in the order the layers
compute. To test this, we put a NaN into one weight of the third residual
block, the module at index 3 of the stack:

In [ ]:
class Checked(tf.keras.Model):
    def __init__(self, layers):
        super().__init__()
        self.seq, self.first_bad = layers, None  # a plain list is tracked

    def call(self, X):
        for layer in self.seq:
            X = layer(X)
            if self.first_bad is None and not bool(
                    tf.reduce_all(tf.math.is_finite(X))):
                self.first_bad = layer.name
        return X

checked = Checked([dense(64)]
                  + [ResidualBlock(64, name=f'block{k}') for k in range(1, 9)]
                  + [dense(10)])
_ = checked(tf.random.normal((2, 20)))  # build the weights
checked.seq[3].body.layers[0].kernel[0, 0].assign(float('nan'))  # break one
_ = checked(tf.random.normal((2, 20)))
print('first non-finite output in', checked.first_bad)

The report names `block3`, the block that holds the NaN. Because the check
is part of `call`, it runs on every forward pass until we edit it out. For a
temporary search, `tf.debugging.enable_check_numerics()` checks the output
of every operation and reports the first one that produces an infinity or a
NaN.

A planted NaN is the easy case, since we knew where to look. In a real run
the finder names the first symptom, and the cause lies at or before it:
activations may grow over several layers before one of them overflows.

### Gradient Inspection

Gradients can be inspected in the same way. The forward signal grew with
depth. Does the backward signal do the same? We record the norm of the
gradient with respect to the output of each residual block, and the
gradient norms of the first linear layer of the third residual block. As a
loss we take the mean squared output of the stack on a minibatch of 256
standard normal inputs.

TensorFlow needs no backward hooks: `tf.GradientTape` returns the gradient
with respect to any tensor computed under it, as a value. The functional
model `probe` returns every block's output, so the tape gives the gradients
with respect to these outputs and to the variables in one call.

In [ ]:
with tf.GradientTape() as tape:
    *acts, out = probe(X)  # every layer's output, then the stack's
    loss = tf.reduce_mean(out ** 2)
block_grads, grads = tape.gradient(
    loss, [acts[1:], net.trainable_variables])
print('block output gradients:',
      ' '.join(f'{float(tf.norm(g)):.3g}' for g in block_grads))
kernel, bias = grads[10:12]  # block 3, first layer
print(f'kernel {float(tf.norm(kernel)):.3g}, bias {float(tf.norm(bias)):.3g}')

Read from the last block back to the first, the gradient norm grows by a
factor of roughly 1.2 to 1.4 per block, five- to sixfold over the stack.
This mirrors the forward growth of the activations, 1.22 per block, and has
the same cause: on the way back, too, each block adds the signal through
its body to the signal that bypasses it.

The two norms of a single layer mean little in isolation. They become
informative when compared across depth or across training steps, which the
exercises do.

What none of these tools can do is make two different machines agree, or
decide whether a difference between two runs matters; for that we compare
results across seeds. Nor did we carry out the whole procedure with which
the section opened, replaying a failed run from a checkpoint
(that section) with the finder attached, though all its parts
are now at hand.

The 8-billion-parameter model of the chapter introduction has not become
any smaller: its weights still take 32 GB in fp32. But we can now say how
such a model is built, initialized, stored, placed on a device, and
observed, each time by a walk over the same tree of modules. The chapters
that follow take these tools for granted, beginning with the convolutional
networks of that section.

## Summary

Every random draw in a training run comes from a generator fixed by a seed.
Seeding each generator the run uses fixes the draws, and private generator
objects or explicit keys keep unrelated draws from shifting them. Seeding
does not fix the arithmetic: floating-point addition depends on the order
of the terms, and some GPU operations add in a varying order, so bitwise
repetition also needs deterministic operations and holds only on one
machine and one software version. It is a tool for debugging; scientific
conclusions should hold across seeds. Hooks, or their equivalents in each
framework, observe the outputs and gradients of every layer of an unmodified
model, which turns the question of where the first NaN appeared into one
forward pass.

## Exercises

1. [code] **Block ablation.** Keras has no hook mechanism that could
   replace a layer's output.
    1. Zero the output of one residual block's body by editing the `call`
       of `ResidualBlock`, which turns the block into the identity, and
       measure the relative change in the output of a freshly built
       `Checked` model on a fixed batch. Repeat for every
       block, and rank the blocks by this change.
    1. Rebuild the model three more times with new random weights and
       repeat the ranking. State whether the conclusion "block $k$ matters
       most" survives, and what evidence such a claim would need.
2. [code] **Residual variance budget.** Override `call` in
   `ResidualBlock` so that it stashes the
   variance of its input, of its body's output, and of its own output, and
   run a freshly built `Checked` model on one batch. Let $\mathbf{x}_k$,
   $\mathbf{b}_k$ and $\mathbf{h}_k$ denote these three tensors for block
   $k$, with each variance taken over all entries. The standard deviations
   in the table of that section grow because each
   block adds its body's output to the stream. Measure how closely

   $$\textrm{Var}[\mathbf{h}_k] = \textrm{Var}[\mathbf{x}_k] + \textrm{Var}[\mathbf{b}_k]$$

   holds for every block, and name the term this equation omits.
3. [code] **Seeds and data loading.** ● Work in a fresh process, since the
   section has already called `enable_op_determinism`, which cannot be
   undone. Build a `tf.data` pipeline over `Dataset.range(64)` that
   shuffles with `shuffle(64, seed=0)` and adds noise to each element in a
   `map` with `num_parallel_calls=4`, and start every run with
   `tf.keras.utils.set_random_seed(0)`.
    1. Draw the noise from NumPy's global generator through
       `tf.numpy_function`. Predict whether two runs attach the same noise
       to the same element under the default `deterministic=True`, and
       justify the prediction from where the randomness is drawn. Verify
       over five runs.
    1. Repeat the five runs twice more: once with
       `tf.config.experimental.enable_op_determinism()` called at program
       start, and once without it but with the noise drawn by
       `tf.random.stateless_normal`, seeded from the element's value.
       Explain each outcome.
4. [code] **Divergence monitor.** ● Turn the finiteness check of
   that section into a per-step training monitor. Rebuild
   the `Checked` model and train it with SGD at learning rate 1 on the
   mean squared error between its output and standard normal targets,
   using one fixed batch of 256 standard normal inputs. Report the first
   step at which a layer's output is non-finite, and the first such layer
   by depth. Check the loss at every step as well, and report whether it
   becomes non-finite before any layer output does.
5. [code] **Gradient norms.** ● Rebuild `net`, multiply the kernel of
   every `Dense` layer inside its residual blocks by 3, and use the mean
   squared output on a fixed batch as the loss.
    1. Compute the norm of each block's output gradient as in this
       section, and each block's parameter-gradient norm from the
       gradients of the trainable variables. Plot both families of norms
       against depth.
    1. Express the gradient of a linear layer's kernel through the layer's
       input and its output gradient. Use this expression to explain why
       the two depth profiles differ, and state which family to monitor for
       an exploding backward signal.

[Discussions](https://d2l.discourse.group/t/36759)